## CloudWatch Metrics and Dashboards

Welcome to the next step in your AWS developer journey! So far, you have learned how to keep your applications secure by managing credentials and protecting your APIs. Now, we will focus on another critical aspect: monitoring your applications.

Monitoring is essential for both security and reliability. It helps you answer questions like:

* Is my application working as expected?
* Are there any unusual patterns or errors?
* How is my application performing over time?

AWS CloudWatch is a service that helps you collect, view, and analyze data from your applications. In this lesson, you will learn how to send your own custom data (called metrics) to CloudWatch and create dashboards to visualize this information. By the end, you will be able to track important numbers from your application and see them in real time.

---

## Quick Recall: Using `boto3` with AWS Services

Before we dive in, let's quickly remind ourselves about boto3. In previous lessons, you used boto3 to interact with AWS services like Secrets Manager and Lambda. boto3 is the official AWS SDK for Python, and it allows you to connect to AWS services directly from your code.

For example, to use a service, you typically create a client like this:

```python
import boto3

# Create a client for AWS CloudWatch
cw = boto3.client('cloudwatch')
```

This `cw` object lets you call CloudWatch functions from your Python code. You will use this same approach to send metrics and create dashboards in this lesson.

---

## Understanding the CloudWatch Flow

Before we start coding, let's visualize how the pieces fit together:

```text
┌─────────────────────┐
│  Python Application │
│   (Your Code)       │
└──────────┬──────────┘
           │
           │ boto3.client('cloudwatch')
           │ put_metric_data()
           ↓
┌─────────────────────┐
│   AWS CloudWatch    │
│  (Stores Metrics)   │
└──────────┬──────────┘
           │
           │ Visualizes
           ↓
┌─────────────────────┐
│  CloudWatch         │
│  Dashboard          │
└─────────────────────┘
```

This diagram shows the complete flow:

1. Your Python application sends metrics to CloudWatch using boto3
2. CloudWatch receives and stores these metrics
3. A dashboard displays the metrics visually for monitoring

> **Important note about timing:** When you send metrics to CloudWatch, they don't appear instantly. It can take several minutes (typically 2-5 minutes) for your metrics to show up in the CloudWatch console or on dashboards. This is normal behavior—CloudWatch processes and aggregates data in the background. So if you don't see your metrics right away, be patient and refresh the dashboard after a few minutes.

Now let's implement this flow step by step.

---

## Emitting Custom Metrics to CloudWatch

A metric is just a number that tells you something about your application. For example, you might want to track the value of each order placed in your app.

Let's see how to send a custom metric to CloudWatch step by step.

### Step 1: Import Required Libraries

First, you need to import the libraries you will use. You will need `boto3` for AWS and some standard libraries for working with time and random numbers.

```python
import time
import random
import boto3
```

* `time` helps you add timestamps to your metrics.
* `random` is used here to generate example data.
* `boto3` is for connecting to AWS CloudWatch.

### Step 2: Create a CloudWatch Client

Next, create a CloudWatch client using boto3:

```python
cw = boto3.client('cloudwatch')
```

This line sets up the connection to CloudWatch.

### Step 3: Define a Function to Send a Metric

Now, let's write a function that sends a single metric value to CloudWatch. We will call this metric `OrderValue`.

```python
def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace="App/Metrics",
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            "Timestamp": time.time(),
            "Value": value
        }]
    )
```

Let's break down what's happening here:

* `Namespace` is a way to group related metrics. Here, we use `"App/Metrics"`.
* `MetricName` is the name of the metric, in this case, `"OrderValue"`.
* `Dimensions` are extra labels to help you filter or group your data. Here, we label the metric with `"Service": "OrdersAPI"`.
* `Timestamp` records when the metric was sent.
* `Value` is the actual number you want to track.

### Step 4: Emit Some Example Metrics

Let's send a few example metrics to CloudWatch. We'll use random numbers to simulate order values.

```python
for i in range(5):
    value = random.uniform(10, 200)
    put_metric_value(value)
    print(f"Emitted metric: OrderValue = {value:.2f}")
    time.sleep(1)
```

This code will:

1. Generate a random order value between 10 and 200.
2. Send it to CloudWatch using your function.
3. Print out what was sent.
4. Wait one second before sending the next value.

**Example output:**

```text
Emitted metric: OrderValue = 153.27
Emitted metric: OrderValue = 45.12
Emitted metric: OrderValue = 189.03
Emitted metric: OrderValue = 77.56
Emitted metric: OrderValue = 120.44
```

Now, you have sent custom metrics to CloudWatch!

---

## Creating and Configuring a CloudWatch Dashboard

A dashboard in CloudWatch is a visual display of your metrics. It helps you see trends and spot problems quickly.

Let's see how to create a dashboard step by step.

### Step 1: Import the JSON Library

You will need the `json` library to build the dashboard configuration.

```python
import json
```

### Step 2: Define a Function to Create a Dashboard

Now, let's write a function that creates a dashboard and adds a widget to show your `OrderValue` metric.

```python
def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName="App-Overview", 
        DashboardBody=json.dumps({"widgets":[widget]})
    )
```

Here's what's happening:

* We define a `widget` that tells CloudWatch what to display.
* The widget shows the average value of the `OrderValue` metric for the `OrdersAPI` service.
* The dashboard is named `"App-Overview"`.
* The dashboard is created or updated using `cw.put_dashboard`.

### Step 3: Create the Dashboard

Now, call the function to create the dashboard:

```python
create_dashboard()
print("Dashboard 'App-Overview' created successfully!")
```

**Example output:**

```text
Dashboard 'App-Overview' created successfully!
```

You can now go to the AWS CloudWatch console and see your dashboard with the `OrderValue` metric displayed.

---

## Summary and Practice Preview

In this lesson, you learned how to:

* Send custom metrics from your application to AWS CloudWatch using Python and boto3.
* Create a CloudWatch dashboard to visualize your metrics in real time.

These skills help you keep an eye on your application's health and performance, which is important for both security and reliability.

Next, you will get hands-on practice by emitting your own metrics and building dashboards in the CodeSignal environment. Remember, on CodeSignal, the required libraries are already installed, so you can focus on writing and running your code.

Great job making it this far! You are now ready to monitor your own AWS applications and gain valuable insights from your data.

## Complete Your First CloudWatch Metric

Now that you understand how CloudWatch metrics work, it's time to put your knowledge into practice! You have been given a partially working `put_metric_value` function that already has the correct namespace, metric name, and dimensions set up. However, two important pieces are missing from the metric data.

Your objective is to complete the function by adding the missing fields:

* Add the `timestamp` field to record when the metric was sent.
* Add the `value` field to include the actual metric data.

Look for the TODO comments in the `put_metric_value` function — they will guide you to exactly where you need to add the missing code. The timestamp should use `time.time()` to get the current time, and the value should use the `value` parameter that is passed to the function.

Once you complete the function, the code will emit five sample metrics to CloudWatch and create a dashboard to visualize them. This hands-on practice will help you master the essential skill of sending custom metrics to monitor your applications effectively.

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            # TODO: Add the Timestamp field using time.time()
            # TODO: Add the Value field using the value parameter
        }]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

Here is the completed `put_metric_value` function with `Timestamp` and `Value` added:

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            "Timestamp": time.time(),
            "Value": value
        }]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

## Adding Metric Dimensions for Better Organization

## Batching Multiple Metrics Efficiently

## Fix Broken Dashboard Widget Configuration

## Build Complete Monitoring Application